## Selling Stocks

using the dynamic programming thinking procss. for some harder problems, the DP attribute would be more obvious

In [1]:
from typing import List, Optional
import math

<b> 121 Best time to buy and sell stock </b>

only buy and sell once, each at different days. it is the keep track of the min value encountered previously, then calculate the difference, and using a global var to check the result.

only 1 transaction

In [ ]:
def maxProfit(prices: List[int]) -> int:
    min_val = float('inf')
    res = 0
    for p in prices:
        min_val = min(min_val, p) # find the lowest price so far
        res = max(res, p-min_val) # first res is 0
    return res

<b> 122 best time to buy and sell stock II </b>

able to buy and sell stock at the same day, so it turns into calculting the largest cumulative price difference when looping over the array.

even though only 1 transaction, the problem allows to sell and buy and the same day, so it is the same as unlimited transactions

In [2]:
def maxProfit(prices: List[int]) -> int:
    # because the stock can be bought and sold at the same day
    # whenever encoutering a stock with higher value, we can sell it
    # otherwise need to determine if the next higher value should be the selling price or not.
    # the sum of all sells will be the max profits.
    res = 0
    for i in range(len(prices)):
        if i > 0 and prices[i] > prices[i-1]:
            res += prices[i] - prices[i-1]
    return res

In [3]:
maxProfit([7,1,5,3,6,4])

7

<b> 123 Best time to buy and sell stock III </b>

At most k transaction, must buy before sell so sequential operations.

In [4]:

def maxProfit(prices: List[int]) -> int:
    # max two transactions,
    # cannot buy and sell at the same time, so cannot just simly sell a stock if the price is higher than previous buying price

    n = len(prices)
    if not prices:
        return 0
    max_transaction = 2

    # construct a matrix with shape [3, n]
    # row index is the max transactin number
    # dp[k][i] means the max profit for ith day if max transaction is k
    # the final result would be dp[-1][-1]

    # initialize the value to 0. for first row, it means max profit for each day when no transaction happen
    # so the value will be 0 for the whole row
    dp = [[0]*n for _ in range(max_transaction + 1)]

    # dp[k][i] = max(for k_th most transaction if not sell at position i, for k_th most transaction, if sell at position i)
    # if not selling at position i, the max profit will be the same as dp[k][i-1] 
    # if want to sell at position i, the buying position must be before position i
    # suppose buying at jth position, j < i, which is prices[i] - prince[j] + dp[k-1][j-1]
    # because prices[i]-princes[j] takes 1 transaction, the profit from previous optimal transaction can only be from row k-1
    # and j is the buying position, so dp[k-1][j-1] 
    # for prices[i] - prince[j] + dp[k-1][j-1], prices[i] is not changed when looping over each position
    # but we can use a local max_v variable to keep check of the the j th position that previously looped over
    # to avoid having additional loop on var j, j's max value can be i, prices[i] - princes[i] + dp[k-1][i] means the largest profix if at most taking k-1 transaction.

    for k in range(1, max_transaction+1):
        max_v = -prices[0]
        for i in range(1, n):
            # cannot sell at first position since not buying any stock yet
            max_v = max(max_v, -prices[i] + dp[k-1][i-1])
            dp[k][i] = max(dp[k][i-1], prices[i] + max_v)
    return dp[-1][-1]


the below one is better, with detailed exlanation. first explain the dp idea, then because ith depends on i-1 th day, the dp can be optimized with fewer dimension

In [5]:

def maxProfit(prices: List[int]) -> int:
    """
    max two transactions
    - t[k][i][0]: the max profit when with k transaction and ith day, after action no stock hold at hand
    - t[k][i][1]: the max profit when with k transaction and ith day, after action 1 stock hold at hand
    k = 2
    * buy + sell is considered as 1 transaction, and transaction is counted when buy action happen
    t[2][i][0] = max(t[2][i-1][0], t[2][i-1][1] + p)
    t[2][i][1] = max(t[2][i-1][1], t[1][i-1][0] - p)
    t[1][i][0] = max(t[1][i-1][0], t[1][i-1][1] + p)
    t[1][i][1] = max(t[1][i-1][1], t[0][i-1][0] - p) = max(t[1][i-1][1], - p) 
    ith transaction only depends on i-1 day
    """
    max_transaction = 2
    # dp_0[k]: max profit at k transaction ith day, no stock hold at hand
    # dp_1[k]: max profit at k transaction ith day, 1 stock hold at hand
    # dp_0: 0 stock for different transaction, so initialize the profit to be 0
    # dp_1: 0 transaction but 1 stock, which is impossible so -inf
    dp_0 = [0 for _ in range(max_transaction + 1)]
    dp_1 = [-math.inf for _ in range(max_transaction + 1)]

    for p in prices:
        for k in range(1, max_transaction+1):
            # dp[k][0] dp[k][1] in right side are from previous day: i-1 th day
            dp_0[k] = max(dp_0[k], dp_1[k]+p)
            dp_1[k] = max(dp_1[k], dp_0[k-1] - p)
    return dp_0[2]


    

<b> 188 best time to buy and sell stock IV </b>

at most K transaction (previous problem is 2 transactions), transactions happens sequentially.

need to consider the edge case then K > n//2, then it can transform into problem 122, unlimited transactions.

In [6]:

def maxProfit(k: int, prices: List[int]) -> int:
    n = len(prices)
    if n <=1:
        return 0
    
    if k >= n/2:
        # all stock in prices list are available for transactions
        t_i_0 = 0 # max profit for ith day and after action no stock hold at hand
        t_i_1 = -math.inf # max profit for ith day and after action 1 stock hold at hand
        for p in prices:
            t_i_0 = max(t_i_0, t_i_1 + p)
            # can buy and sell at the same day, so no need to check t_(i-1)_0
            t_i_1 = max(t_i_1, t_i_0 - p)
        return t_i_0
    
    # for other cases
    # for different k transactions, largest profit when no stock hold at hand for each ith day
    dp_0 = [0] * (k+1)
    # for different k transactions, largest profit when 1 stock hold at hand for each ith day
    dp_1 = [-math.inf] * (k+1)

    for p in prices:
        for k in range(1, k+1):
            # either no action or sell the stock which is coming from t_k_(i-1)_1
            # because i only rely on i-1th record, here using one same var
            dp_0[k] = max(dp_0[k], dp_1[k] + p)
            # when selling stock, it requies to use the value from previous transaction
            # because one buy+sell is one transaction, and k is updated at buy step
            dp_1[k] = max(dp_1[k], dp_0[k-1] - p)
    return dp_0[k]



    

<b> 309 best time to buy and sell stock with cooldown </b>

no limitations on transaction times, and sequetial transactions on different days, but with cool down period:
- after selling stock, cannot buy stock on the next day


In [7]:

def maxProfit(prices: List[int]) -> int:
    """
    t[i][k][0]: max profit at ith day for transaction k when no stock hold at hand after action
    t[i][k][1]: max profit at ith day for transaction k when 1 stock hold at hand after action
    
    t[i][k][0] = max(t[i-1][k][0], t[i-1][k][1] + p) # no action, or sell stock
    t[i][k][1] = max(t[i-1][k][1], t[i-2][k-1][0] -p) # no action, or buy stock but with cool down restriction
    
    unlimited transactions, so k is not a restriction
    """

    dp_cooldown = 0
    dp_sell = 0
    dp_buy = -math.inf # initial value,

    for p in prices:
        max_profit_from_previous_sell = dp_sell
        dp_sell = max(dp_sell, dp_buy + p)
        dp_buy = max(dp_buy, dp_cooldown - p)
        dp_cooldown = max_profit_from_previous_sell
    return dp_sell


    

the above solution is a simplication of the following one. the current day transaction only depends on previous day transaction, so the dp array is simplied into a single variable.

cool down (or reset) is followed by sell action, so the max profit is actually always the previous max profit at sell action. Because in previous solution the dp_sell is updated to current day, and lose the previous day's value, so it uses a tmp var to store it then assign it to the cool_down value

In [8]:

def maxProfit(prices: List[int]) -> int:
    # 3 states, buy, sell, reset(cool down)
    l = len(prices)
    # base case, 1 element, no sell, no buy
    if l == 1:
        return 0
    buy = [0 for _ in range(l)]
    sell = [0 for _ in range(l)]
    reset = [0 for _ in range(l)]

    buy[0] = - prices[0]
    sell[0] = - math.inf
    reset[0] = 0


    for i in range(1, l):
        # next buy step: either no action -> inherite previous buy state, or come from reset and literally buy current stock
        buy[i] = max(buy[i-1], reset[i-1]-prices[i])
        # next sell step: either no action -> inherite prevous sell state (already include the cool down situation, see how reset state is updated), or sell stock at price[i] after buy state
        sell[i] = max(sell[i-1], buy[i-1]+prices[i])
        # next reset step: cool down from cool down, or buy, or sell
        reset[i] = max(reset[i-1], sell[i-1])
    
    return max(reset[-1], sell[-1])
    

<b> 714 Best time to buy and sell stock with transaction fee </b>

can complete as many transactions as needed so cumulative profit. But each profit has a transaction fee (one buy and sell considered one transaction).   
here the fee is charged at buy time. (it still work if charge at selling time)



In [9]:
import math

def maxProfit(prices: List[int], fee: int) -> int:
    """
    this is the test if pay fee at stock buying time still work
    """
    # with 1 stock after transaction, but at time 0 it cant happen so initialize using -math.inf
    dp_buy = -math.inf 
    # with 0 stack after transaction, at time 0 nothing happen and can be initialize as 0
    dp_sell = 0

    for p in prices:
        dp_buy = max(dp_buy, dp_sell-p-fee)
        dp_sell = max(dp_sell, dp_buy+p)
    return dp_sell

<b> 2012 Sum of beauty in the array </b>

at position i,  check if max_val_from_left < value[i] < min_val_from_right, otherwise check if the value[i] is greater than its neighbors.

In [10]:

def sumOfBeauties(nums: List[int]) -> int:
    # beauty of value equals to 2 when nums[i] < all elements in the right
    # same as nums[i] < min value from the right sublist
    # and when nums[i] > all elements in the left, same as nums[i] > max value from the left sublist

    n = len(nums)
    # n is guaranteed to >=3, otherwise check n's size
    min_from_right = [math.inf] * n
    min_from_right[-1] = nums[-1]
    max_from_left = [-math.inf] * n
    max_from_left[0] = nums[0]

    for i in range(n-2, -1, -1):
        min_from_right[i] = min(min_from_right[i+1], nums[i])

    # can update the max_from_left list in a separate loop or optimize by using one same loop when calculting the sum of beauty values
    res = 0
    for i in range(1, n-1):
        if max_from_left[i-1] < nums[i] < min_from_right[i+1]:
            res += 2
        elif nums[i-1] < nums[i] < nums[i+1]:
            res += 1
        # update max_from_left
        max_from_left[i] = max(max_from_left[i-1], nums[i])
    return res


<b> 2291 maximum profit from trading stocks </b>

You are given two 0-indexed integer arrays of the same length present and future where present[i] is the current price of the ith stock and future[i] is the price of the ith stock a year in the future. You may buy each stock at most once. You are also given an integer budget representing the amount of money you currently have.

Return the maximum amount of profit you can make.


the second loop is in reverse option, from budget to p. If in the order of from p to budget, dp[j] for the current stock may be overwrite by dp[j-p] value later, which leads to using the same stock transaction updating the profits multiple times.

by looping in reverse, it ensures that the dp[j] value from a previous stock is preserved while updating for the current stock.

In [11]:

def maximumProfit(present: List[int], future: List[int], budget: int) -> int:
    # dp[i] max profit when using budget value i
    dp = [0]*(budget+1)
    for p, f in zip(present,future):
        # for stock at i th day, the valid budget will be p to dudget
        for j in range(budget, p-1, -1):
            # either dont do anything, or buy/sell the stock
            # corresponding the budget should be used to pay the price
            dp[j] = max(dp[j], dp[j-p] + f - p)
    return dp[-1]

<b> 1014 Best sightseeing pair </b>
the problem wants to know the max value of A[i]+A[j]+i-j where i < j  
and it can be transformed as A[i]+i + A[j]-j   
so keep track of the best value of A[i]+i while looping over the array.

time complexity is O(N) N is the length of the array  
space complexity is O(1)



In [12]:

def maxScoreSightseeingPair(values: List[int]) -> int:
    """
    the problem wants to know the max value of
    A[i]+A[j]+i-j = A[i]+i + A[j]-j where i < j
    so keep track of the best value of A[i]+i
    """
    res = 0
    best_prev_value_with_index = 0
    for ind, val in enumerate(values):
        res = max(res, best_prev_value_with_index + val - ind)
        best_prev_value_with_index = max(best_prev_value_with_index, val+ind)
    return res

<b> 1769 Minimum number of operations to move all balls to each box </b>

similar to the stock selling problems, using a dynamic programming ways that next operation depends on previous operation's value.

In [13]:

def minOperations(boxes: str) -> List[int]:
    """
    time complexity: O(N)
    space complexity: O(N)

    if thinking accum_count and operation_count as a list in size n
    in the loop from left to right:
    accum_count[i] means how many balls so far from pos 0 to pos i (i included)
    operation_count[i] means how many steps needed to move all balls encountered before (from position 0 to i-1) to pos i
    operation_count[i] = operation_count[i-1] + accum_count[i-1]
    operation_count[0] = 0
    mean: all the balls seen before the pos i-1 are moved to pos i-1 -> operation_count[i-1], plus how many balls seen at position i-1, those ball each takes one step to move from i-1 to i

    because value at index i only depends on i-1, the var is optimized from list to a single value.
    """
    n = len(boxes)
    res = [0]*n
    accum_count = 0
    operation_count = 0
    
    for i in range(n):
        res[i] += operation_count
        accum_count += boxes[i]=="1"
        operation_count += accum_count

    accum_count = 0
    operation_count = 0
    for i in range(n-1, -1, -1):
        res[i] += operation_count
        accum_count += boxes[i]=="1"
        operation_count += accum_count
    return res


a brute force way is to loop over each index and calculate the sum of distance of moving all balls to this position, which takes O(N*K) time complexity where K is the number of position having ball

In [14]:

def minOperations(boxes: str) -> List[int]:
    """
    find the index where having box -> a new location_array
    then loop over boxes array index to calculate the distance against all the items in location_array
    """

    location_array = [i for i, v in enumerate(boxes) if v=="1"]
    res = []
    for i, v in enumerate(boxes):
        res.append(sum(abs(i- source_loc) for source_loc in location_array))
    return res

<b> 2270 number of ways to split array </b>

my first solution is to keep an accumulative sum of the array from the right, then compare the left part accumulative sum while looping over the array, but this takes O(n) space.  when having the accumlative sum of left part, the right sum is actually = total_sum - left_accumulative_sum, so the space complexity can be O(1)

In [15]:

def waysToSplitArray(nums: List[int]) -> int:
    """
    calculate the sum of the whole array

    then when loop over the array, calculate the accumulative sum, the sum of its right part is actually whole_array_sum - left_accumulative_sum

    time complexity : O(N)
    space complexity: O(1)
    """
    n = len(nums)
    if n < 2:
        return 0
    all_sum = sum(nums)
    acc_sum = nums[0]
    res = 0
    for i in range(1, n):
        if acc_sum >= all_sum - acc_sum: res += 1
        acc_sum+=nums[i]
    return res